# Measles Risk Model v3: Temporal Validation

## Key Improvements Over v2:

1. **2025 Data Only** - Focus on the outbreak year
2. **Temporal Validation** - Train on early 2025 (Jan-Jun), test on late 2025 (Jul-Dec)
3. **Spatial Features** - Distance to nearest outbreak county
4. **Proper Calibration** - Risk thresholds based on actual prevalence

## Validation Approach:

```
Timeline: Jan 2025 -------- Jun 2025 -------- Dec 2025
          |                 |                 |
          |<-- TRAINING -->|<-- VALIDATION -->|
          |                 |                 |
          Counties with     Predict which     Check if
          cases by Jun      counties get      prediction
                            cases Jul-Dec     was correct
```

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_recall_curve, roc_curve, classification_report,
                             confusion_matrix)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries loaded")

In [ ]:
# =============================================================================
# LOAD DATA
# =============================================================================
measles_raw = pd.read_csv('measles_timeseries.csv')
mmr_raw = pd.read_csv('MMR_2026 - mmr.csv')

# Parse dates
measles_raw['date'] = pd.to_datetime(measles_raw['date'], format='%m/%d/%y', errors='coerce')

# Filter to 2025 ONLY
measles_2025 = measles_raw[measles_raw['date'].dt.year == 2025].copy()
print(f"Total 2025 records: {len(measles_2025)}")
print(f"Date range: {measles_2025['date'].min()} to {measles_2025['date'].max()}")

In [ ]:
# =============================================================================
# DEFINE TEMPORAL SPLIT
# =============================================================================

# Split point: June 30, 2025
SPLIT_DATE = pd.Timestamp('2025-06-30')

# Training period: Jan 1 - Jun 30, 2025
# Validation period: Jul 1 - Dec 31, 2025

train_period = measles_2025[measles_2025['date'] <= SPLIT_DATE].copy()
valid_period = measles_2025[measles_2025['date'] > SPLIT_DATE].copy()

print(f"Training period: {train_period['date'].min()} to {train_period['date'].max()}")
print(f"Training records: {len(train_period)}")
print(f"\nValidation period: {valid_period['date'].min()} to {valid_period['date'].max()}")
print(f"Validation records: {len(valid_period)}")

In [ ]:
# =============================================================================
# STANDARDIZE FIPS CODES
# =============================================================================

def standardize_fips(fips):
    try:
        return str(int(float(fips))).zfill(5)
    except:
        return None

# Apply to all datasets
measles_2025['fips_std'] = measles_2025['fips'].apply(standardize_fips)
train_period['fips_std'] = train_period['fips'].apply(standardize_fips)
valid_period['fips_std'] = valid_period['fips'].apply(standardize_fips)
mmr_raw['fips_std'] = mmr_raw['FIPS'].apply(standardize_fips)

# Filter to county-level only
train_county = train_period[train_period['vaccination granularity'] == 'county'].copy()
valid_county = valid_period[valid_period['vaccination granularity'] == 'county'].copy()

print(f"Training counties with cases: {train_county['fips_std'].nunique()}")
print(f"Validation counties with cases: {valid_county['fips_std'].nunique()}")

In [ ]:
# =============================================================================
# IDENTIFY TRAINING vs VALIDATION COUNTIES
# =============================================================================

# Counties that had cases by June 30 (KNOWN at training time)
counties_cases_by_jun = set(train_county['fips_std'].dropna().unique())

# Counties that got NEW cases Jul-Dec (what we want to PREDICT)
counties_cases_jul_dec = set(valid_county['fips_std'].dropna().unique())

# NEW outbreak counties = got cases in Jul-Dec but NOT before
new_outbreak_counties = counties_cases_jul_dec - counties_cases_by_jun

print(f"Counties with cases by Jun 30: {len(counties_cases_by_jun)}")
print(f"Counties with cases Jul-Dec: {len(counties_cases_jul_dec)}")
print(f"NEW outbreak counties (Jul-Dec): {len(new_outbreak_counties)}")

In [ ]:
# =============================================================================
# GET LOCATIONS OF OUTBREAK COUNTIES (for spatial features)
# =============================================================================

# Get lat/lon for counties with cases by June
outbreak_locations = train_county.groupby('fips_std').agg({
    'latitude': 'first',
    'longitude': 'first',
    'cases_total': 'max',
    'county': 'first',
    'state': 'first'
}).reset_index()

outbreak_locations = outbreak_locations.dropna(subset=['latitude', 'longitude'])
print(f"Outbreak counties with coordinates: {len(outbreak_locations)}")
print(f"\nTop outbreak counties by June 30:")
print(outbreak_locations.nlargest(10, 'cases_total')[['county', 'state', 'cases_total']])

---
## Process MMR Data

In [ ]:
# =============================================================================
# MMR FEATURE ENGINEERING
# =============================================================================

mmr_year_cols = ['SY2024_25', 'SY2023_24', 'SY2022_23', 'SY2021_22',
                 'SY2020_21', 'SY2019_20', 'SY2018_19', 'SY2017_18']

# Convert to numeric
for col in mmr_year_cols:
    mmr_raw[col] = pd.to_numeric(mmr_raw[col], errors='coerce')

def get_mmr_features(row):
    rates = [row[col] for col in mmr_year_cols if pd.notna(row[col])]
    if len(rates) == 0:
        return pd.Series({'latest_mmr': np.nan, 'mmr_trend': 0, 'mmr_min': np.nan, 'mmr_std': 0})
    return pd.Series({
        'latest_mmr': rates[0],
        'mmr_trend': rates[0] - rates[-1] if len(rates) >= 2 else 0,
        'mmr_min': min(rates),
        'mmr_std': np.std(rates) if len(rates) >= 2 else 0  # Volatility in coverage
    })

mmr_features = mmr_raw.apply(get_mmr_features, axis=1)
mmr_df = pd.concat([mmr_raw[['fips_std', 'County', 'State']], mmr_features], axis=1)
mmr_df = mmr_df.rename(columns={'County': 'county', 'State': 'state'})

print(f"Counties with MMR data: {mmr_df['latest_mmr'].notna().sum()}")

---
## Build Features (using ONLY information available at training time)

In [ ]:
# =============================================================================
# SPATIAL FEATURES: Distance to Nearest Outbreak
# =============================================================================

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in km between two points."""
    R = 6371  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def calculate_spatial_features(county_fips, county_lat, county_lon, outbreak_df):
    """
    Calculate spatial features for a county based on proximity to outbreaks.
    
    Returns:
    - min_distance_to_outbreak: km to nearest outbreak county
    - outbreak_exposure_100km: weighted case count within 100km
    - outbreak_exposure_300km: weighted case count within 300km
    - num_outbreaks_100km: count of outbreak counties within 100km
    """
    if pd.isna(county_lat) or pd.isna(county_lon) or len(outbreak_df) == 0:
        return pd.Series({
            'min_distance_to_outbreak': 9999,
            'outbreak_exposure_100km': 0,
            'outbreak_exposure_300km': 0,
            'num_outbreaks_100km': 0
        })
    
    distances = []
    exposure_100 = 0
    exposure_300 = 0
    count_100 = 0
    
    for _, outbreak in outbreak_df.iterrows():
        if outbreak['fips_std'] == county_fips:
            continue  # Skip self
        
        d = haversine_distance(county_lat, county_lon, 
                               outbreak['latitude'], outbreak['longitude'])
        distances.append(d)
        
        # Weighted exposure (closer = more exposure)
        cases = outbreak['cases_total']
        if d <= 100:
            exposure_100 += cases / max(d, 1)  # Avoid div by 0
            count_100 += 1
        if d <= 300:
            exposure_300 += cases / max(d, 1)
    
    return pd.Series({
        'min_distance_to_outbreak': min(distances) if distances else 9999,
        'outbreak_exposure_100km': exposure_100,
        'outbreak_exposure_300km': exposure_300,
        'num_outbreaks_100km': count_100
    })

print("Spatial feature functions defined.")

In [ ]:
# =============================================================================
# BUILD MASTER DATASET
# =============================================================================

# Start with MMR data
master_df = mmr_df[['fips_std', 'county', 'state', 'latest_mmr', 'mmr_trend', 'mmr_min', 'mmr_std']].copy()

# Add coordinates from measles data (for spatial features)
all_coords = measles_2025.groupby('fips_std').agg({
    'latitude': 'first',
    'longitude': 'first',
    'population': 'first'
}).reset_index()

master_df = master_df.merge(all_coords, on='fips_std', how='left')

# Mark if county had cases by June 30 (training target = 0, already has outbreak)
master_df['had_cases_by_jun'] = master_df['fips_std'].isin(counties_cases_by_jun).astype(int)

# Mark if county got NEW cases Jul-Dec (validation target)
master_df['got_new_cases_jul_dec'] = master_df['fips_std'].isin(new_outbreak_counties).astype(int)

print(f"Master dataset: {len(master_df)} counties")
print(f"Had cases by June: {master_df['had_cases_by_jun'].sum()}")
print(f"Got NEW cases Jul-Dec: {master_df['got_new_cases_jul_dec'].sum()}")

In [ ]:
# =============================================================================
# CALCULATE SPATIAL FEATURES (using outbreak data as of June 30)
# =============================================================================

print("Calculating spatial features (this may take a moment)...")

spatial_features = master_df.apply(
    lambda row: calculate_spatial_features(
        row['fips_std'], row['latitude'], row['longitude'], outbreak_locations
    ), axis=1
)

master_df = pd.concat([master_df, spatial_features], axis=1)

print("\nSpatial feature statistics:")
print(master_df[['min_distance_to_outbreak', 'outbreak_exposure_100km', 'num_outbreaks_100km']].describe())

In [ ]:
# =============================================================================
# ADD STATE-LEVEL FEATURES (as of June 30)
# =============================================================================

# State outbreak status by June 30
state_stats_jun = train_county.groupby('state').agg({
    'cases_total': 'max',
    'fips_std': 'nunique'
}).rename(columns={
    'cases_total': 'state_cases_by_jun',
    'fips_std': 'state_counties_with_cases_jun'
}).reset_index()

master_df = master_df.merge(state_stats_jun, on='state', how='left')
master_df['state_cases_by_jun'] = master_df['state_cases_by_jun'].fillna(0)
master_df['state_counties_with_cases_jun'] = master_df['state_counties_with_cases_jun'].fillna(0)

# Binary: is there an active outbreak in this state?
master_df['state_has_outbreak'] = (master_df['state_cases_by_jun'] > 0).astype(int)

print(f"States with outbreaks by June: {master_df['state_has_outbreak'].sum()} unique counties in outbreak states")

---
## Model Training with Temporal Validation

In [ ]:
# =============================================================================
# PREPARE TRAINING DATA
# =============================================================================

# EXCLUDE counties that already had cases by June (they're not "at risk" anymore)
# We want to predict: among counties WITHOUT cases by June, which will get cases?

at_risk_counties = master_df[master_df['had_cases_by_jun'] == 0].copy()
print(f"Counties at risk (no cases by June 30): {len(at_risk_counties)}")

# Target: did they get cases Jul-Dec?
print(f"Of these, got cases Jul-Dec: {at_risk_counties['got_new_cases_jul_dec'].sum()}")
print(f"Remained case-free: {(at_risk_counties['got_new_cases_jul_dec']==0).sum()}")

In [ ]:
# =============================================================================
# DEFINE FEATURES
# =============================================================================

FEATURE_COLS = [
    # Vaccination (static risk)
    'latest_mmr',
    'mmr_trend',
    'mmr_min',
    'mmr_std',
    
    # Spatial (proximity to outbreaks as of June)
    'min_distance_to_outbreak',
    'outbreak_exposure_100km',
    'outbreak_exposure_300km',
    'num_outbreaks_100km',
    
    # State context
    'state_cases_by_jun',
    'state_counties_with_cases_jun',
    'state_has_outbreak',
]

# Filter to counties with valid MMR data
model_data = at_risk_counties.dropna(subset=['latest_mmr']).copy()
print(f"Counties with valid features: {len(model_data)}")
print(f"Target distribution: {model_data['got_new_cases_jul_dec'].value_counts().to_dict()}")

In [ ]:
# =============================================================================
# PREPARE X AND Y
# =============================================================================

X = model_data[FEATURE_COLS].copy()
y = model_data['got_new_cases_jul_dec'].copy()

# Impute missing values
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=FEATURE_COLS, index=X.index)

# Scale
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=FEATURE_COLS, index=X.index)

print(f"Feature matrix: {X_scaled.shape}")
print(f"\nFeature summary:")
print(X_imputed.describe().T[['mean', 'std', 'min', 'max']])

In [ ]:
# =============================================================================
# TRAIN MODELS (no train/test split - we're validating temporally)
# =============================================================================

# The ENTIRE model_data is our "test" - counties at risk as of June 30
# We train on ALL features known by June 30 to predict Jul-Dec outcomes

# For honest evaluation, we'll use cross-validation
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, C=0.1, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=5,
                                            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=4, 
                                                     learning_rate=0.05, random_state=RANDOM_STATE)
}

results = {}
for name, model in models.items():
    # Use appropriate features
    X_use = X_scaled if 'Logistic' in name else X_imputed
    
    # Get cross-validated predictions
    y_prob = cross_val_predict(model, X_use, y, cv=cv, method='predict_proba')[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    
    # Fit final model on all data
    model.fit(X_use, y)
    
    results[name] = {
        'model': model,
        'cv_probabilities': y_prob,
        'roc_auc': roc_auc_score(y, y_prob),
        'avg_precision': average_precision_score(y, y_prob),
        'f1': f1_score(y, y_pred)
    }
    print(f"{name}: AUC={results[name]['roc_auc']:.4f}, AP={results[name]['avg_precision']:.4f}, F1={results[name]['f1']:.4f}")

In [ ]:
# =============================================================================
# SELECT BEST MODEL AND ANALYZE
# =============================================================================

best_name = max(results, key=lambda x: results[x]['roc_auc'])
best_model = results[best_name]['model']
best_probs = results[best_name]['cv_probabilities']

print(f"\nBest Model: {best_name}")
print(f"ROC-AUC: {results[best_name]['roc_auc']:.4f}")
print(f"Average Precision: {results[best_name]['avg_precision']:.4f}")

# Feature importance
if hasattr(best_model, 'feature_importances_'):
    importance = pd.DataFrame({
        'feature': FEATURE_COLS,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    print(f"\nFeature Importance:")
    print(importance.to_string(index=False))
elif hasattr(best_model, 'coef_'):
    importance = pd.DataFrame({
        'feature': FEATURE_COLS,
        'coefficient': best_model.coef_[0]
    }).sort_values('coefficient', key=abs, ascending=False)
    print(f"\nLogistic Regression Coefficients:")
    print(importance.to_string(index=False))

In [ ]:
# =============================================================================
# CALIBRATE RISK THRESHOLDS
# =============================================================================

# Use precision-recall to find optimal threshold
precision, recall, thresholds = precision_recall_curve(y, best_probs)

# Find threshold that gives reasonable precision (at least 20%)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_threshold_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[best_threshold_idx] if best_threshold_idx < len(thresholds) else 0.5

print(f"Optimal threshold (max F1): {optimal_threshold:.3f}")
print(f"At this threshold: Precision={precision[best_threshold_idx]:.2%}, Recall={recall[best_threshold_idx]:.2%}")

# Alternative: Use percentile-based thresholds
actual_outbreak_rate = y.mean()
print(f"\nActual outbreak rate: {actual_outbreak_rate:.2%}")

# Set thresholds based on actual prevalence
# Critical: top 5% risk, High: top 15%, Moderate: top 30%, Low: rest
p95 = np.percentile(best_probs, 95)
p85 = np.percentile(best_probs, 85)
p70 = np.percentile(best_probs, 70)

print(f"\nPercentile-based thresholds:")
print(f"  Critical (top 5%): risk > {p95:.3f}")
print(f"  High (top 15%): risk > {p85:.3f}")
print(f"  Moderate (top 30%): risk > {p70:.3f}")

In [ ]:
# =============================================================================
# ASSIGN CALIBRATED RISK CATEGORIES
# =============================================================================

model_data['risk_score'] = best_probs

# Use percentile-based categories (more meaningful than arbitrary thresholds)
def assign_risk_category(score):
    if score >= p95:
        return 'Critical'
    elif score >= p85:
        return 'High'
    elif score >= p70:
        return 'Moderate'
    else:
        return 'Low'

model_data['risk_category'] = model_data['risk_score'].apply(assign_risk_category)

print("Risk Category Distribution (Calibrated):")
print(model_data['risk_category'].value_counts())

# Check precision within each category
print("\nActual outbreak rate by risk category:")
for cat in ['Critical', 'High', 'Moderate', 'Low']:
    subset = model_data[model_data['risk_category'] == cat]
    rate = subset['got_new_cases_jul_dec'].mean()
    print(f"  {cat}: {rate:.2%} ({subset['got_new_cases_jul_dec'].sum()}/{len(subset)} counties)")

---
## Visualization

In [ ]:
# =============================================================================
# PLOT RESULTS
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. ROC Curves
ax1 = axes[0, 0]
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y, res['cv_probabilities'])
    ax1.plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.3f})")
ax1.plot([0,1], [0,1], 'k--', label='Random')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves (Temporal Validation: Jan-Jun → Jul-Dec)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Precision-Recall
ax2 = axes[0, 1]
for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y, res['cv_probabilities'])
    ax2.plot(rec, prec, label=f"{name} (AP={res['avg_precision']:.3f})")
ax2.axhline(y=actual_outbreak_rate, color='red', linestyle='--', label=f'Baseline ({actual_outbreak_rate:.1%})')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Risk Score Distribution
ax3 = axes[1, 0]
for outcome in [0, 1]:
    subset = model_data[model_data['got_new_cases_jul_dec'] == outcome]['risk_score']
    label = 'Got Cases (Jul-Dec)' if outcome == 1 else 'No Cases'
    ax3.hist(subset, bins=30, alpha=0.6, label=label, density=True)
ax3.axvline(x=optimal_threshold, color='red', linestyle='--', label=f'Optimal Threshold ({optimal_threshold:.2f})')
ax3.set_xlabel('Risk Score')
ax3.set_ylabel('Density')
ax3.set_title('Risk Score Distribution by Actual Outcome')
ax3.legend()

# 4. Outbreak Rate by Risk Category
ax4 = axes[1, 1]
cat_order = ['Low', 'Moderate', 'High', 'Critical']
outbreak_rates = [model_data[model_data['risk_category'] == cat]['got_new_cases_jul_dec'].mean() 
                  for cat in cat_order]
colors = ['green', 'yellow', 'orange', 'red']
bars = ax4.bar(cat_order, outbreak_rates, color=colors, edgecolor='black')
ax4.axhline(y=actual_outbreak_rate, color='blue', linestyle='--', label=f'Overall Rate ({actual_outbreak_rate:.1%})')
ax4.set_xlabel('Risk Category')
ax4.set_ylabel('Actual Outbreak Rate (Jul-Dec)')
ax4.set_title('Model Calibration: Predicted vs Actual Outbreak Rate')
ax4.legend()
for bar, rate in zip(bars, outbreak_rates):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{rate:.1%}', 
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('temporal_validation_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Saved: temporal_validation_results.png")

In [ ]:
# =============================================================================
# CONFUSION MATRIX AT OPTIMAL THRESHOLD
# =============================================================================

y_pred_optimal = (best_probs >= optimal_threshold).astype(int)

print("\nClassification Report (at optimal threshold):")
print(classification_report(y, y_pred_optimal, target_names=['No Outbreak', 'Outbreak']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y, y_pred_optimal)
print(f"                 Predicted")
print(f"                 No    Yes")
print(f"Actual No      {cm[0,0]:5d}  {cm[0,1]:5d}")
print(f"Actual Yes     {cm[1,0]:5d}  {cm[1,1]:5d}")

---
## Output: High-Risk Counties

In [ ]:
# =============================================================================
# TOP RISK COUNTIES
# =============================================================================

print("="*80)
print("TOP 20 HIGHEST RISK COUNTIES (as predicted from June 2025 data)")
print("="*80)

top_risk = model_data.nlargest(20, 'risk_score')[[
    'county', 'state', 'risk_score', 'risk_category', 
    'latest_mmr', 'min_distance_to_outbreak', 'got_new_cases_jul_dec'
]].copy()

top_risk['actually_got_cases'] = top_risk['got_new_cases_jul_dec'].map({1: 'YES', 0: 'No'})
print(top_risk.drop(columns=['got_new_cases_jul_dec']).to_string(index=False))

In [ ]:
# =============================================================================
# SAVE OUTPUT
# =============================================================================

output_df = model_data[[
    'fips_std', 'county', 'state',
    'risk_score', 'risk_category',
    'latest_mmr', 'mmr_trend',
    'min_distance_to_outbreak', 'num_outbreaks_100km',
    'state_cases_by_jun', 'state_has_outbreak',
    'got_new_cases_jul_dec'  # For validation
]].copy()

output_df = output_df.sort_values('risk_score', ascending=False)
output_df.to_csv('county_risk_temporal_validation.csv', index=False)

print(f"\n✓ Saved: county_risk_temporal_validation.csv")
print(f"  Counties: {len(output_df)}")

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "="*80)
print("TEMPORAL VALIDATION SUMMARY")
print("="*80)

print(f"\n📅 VALIDATION DESIGN:")
print(f"   Training period: Jan 1 - Jun 30, 2025")
print(f"   Validation period: Jul 1 - Dec 31, 2025")
print(f"   Question: Which counties (no cases by Jun) will get cases by Dec?")

print(f"\n📊 DATA:")
print(f"   Counties at risk (no cases by June): {len(model_data):,}")
print(f"   Counties that got cases (Jul-Dec): {model_data['got_new_cases_jul_dec'].sum()}")
print(f"   Outbreak rate: {actual_outbreak_rate:.2%}")

print(f"\n🎯 MODEL PERFORMANCE ({best_name}):")
print(f"   ROC-AUC: {results[best_name]['roc_auc']:.4f}")
print(f"   Average Precision: {results[best_name]['avg_precision']:.4f}")
print(f"   F1 Score: {results[best_name]['f1']:.4f}")

print(f"\n✅ MODEL CALIBRATION (does risk category predict outbreak rate?):")
for cat in ['Critical', 'High', 'Moderate', 'Low']:
    subset = model_data[model_data['risk_category'] == cat]
    rate = subset['got_new_cases_jul_dec'].mean()
    n = len(subset)
    print(f"   {cat:10s}: {rate:6.2%} outbreak rate ({n:4d} counties)")

print(f"\n📁 OUTPUT FILES:")
print(f"   - county_risk_temporal_validation.csv")
print(f"   - temporal_validation_results.png")